<a href="https://colab.research.google.com/github/sulthanalihsan/data-science-2026/blob/main/Pertemuan6_Muhamad_Sulthan_Al_Ihsan_250401020154.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pertemuan 6 — Persiapan Data
**Nama  :** Muhammad Sulthan Al Ihsan

**NIM   :** 250401020154

**Mata Kuliah:** Data Science —  S1 PJJ Informatika

**Kelas:** IF401

Import Library

In [68]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import (train_test_split)

import pandas as pd

# **COBACOBA**

# **3.1 Label Encoding**
Mengubah kategori menjadi angka unik (0, 1, 2, dst.) berdasarkan urutan alfabet. Metode ini sederhana namun berisiko menciptakan asumsi urutan yang tidak ada pada data nominal.

In [69]:
df = pd.DataFrame({
'Gender': ['male', 'female', 'female', 'male', 'female'],
'Survived': [0, 1, 1, 0, 1]
})
# ── Label Encoding ────────────────────────────────────────────────
le = LabelEncoder()
# fit_transform: belajar pemetaan + langsung transformasi
df['Gender_enc'] = le.fit_transform(df['Gender'])
print(df)
# Hasil: female → 0, male → 1
# Lihat pemetaan kelas
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Mapping:', mapping) # {'female': 0, 'male': 1}
# Decode balik ke label asli
decoded = le.inverse_transform([0, 1, 0])
print(decoded) # ['female' 'male' 'female']

   Gender  Survived  Gender_enc
0    male         0           1
1  female         1           0
2  female         1           0
3    male         0           1
4  female         1           0
Mapping: {'female': np.int64(0), 'male': np.int64(1)}
['female' 'male' 'female']


# **3.2 One-Hot Encoding (OHE)**
Membuat kolom biner terpisah untuk setiap kategori (bernilai 1 jika cocok, 0 jika tidak). Metode ini aman untuk data nominal karena tidak memberikan bobot atau urutan antar kategori.

**3.2.2 Implementasi Python**

In [70]:
df = pd.DataFrame({'City': ['Jakarta','Surabaya','Bandung','Jakarta','Surabaya']})
# ── Cara 1: Pandas get_dummies (paling mudah) ─────────────────────
df_ohe = pd.get_dummies(df,
columns=['City'],
drop_first=False, # True untuk hindari dummy variable trap
dtype=int) # hasilkan 0/1 bukan True/False
print(df_ohe)
# ── Cara 2: sklearn OneHotEncoder (untuk pipeline ML) ─────────────
enc = OneHotEncoder(sparse_output=False, drop='first')
X_enc = enc.fit_transform(df[['City']])
print('Nama fitur:', enc.get_feature_names_out())
# ── Cara 3: Dengan drop_first=True (menghindari Dummy Variable Trap)
df_ohe_drop = pd.get_dummies(df, columns=['City'],
drop_first=True, dtype=int)
# Hasilnya: City_Surabaya, City_Bandung (City_Jakarta dihapus)
# City_Jakarta dapat disimpulkan: jika dua kolom lain = 0, maka Jakarta

   City_Bandung  City_Jakarta  City_Surabaya
0             0             1              0
1             0             0              1
2             1             0              0
3             0             1              0
4             0             0              1
Nama fitur: ['City_Jakarta' 'City_Surabaya']


# **3.3 Ordinal Encoding**
Ordinal Encoding digunakan untuk data kategorikal yang memiliki urutan alami (misalnya: Low < Medium < High). Berbeda dengan Label Encoding yang mengurutkan berdasarkan abjad, metode ini memungkinkan kita menetapkan angka secara manual sesuai makna atau hierarki data yang sebenarnya.

In [71]:
df = pd.DataFrame({
'Pendidikan': ['SMA', 'S1', 'SD', 'D3', 'S2', 'SMP'],
'Gaji_juta': [5, 12, 3, 8, 18, 4]
})
# Definisikan urutan kategori secara eksplisit
edu_order = [['SD', 'SMP', 'SMA', 'D3', 'S1', 'S2']]
enc = OrdinalEncoder(
categories=edu_order,
handle_unknown='use_encoded_value',
unknown_value=-1) # kategori baru → -1
df['Pendidikan_enc'] = enc.fit_transform(df[['Pendidikan']])
print(df.sort_values('Pendidikan_enc'))
# SD=0, SMP=1, SMA=2, D3=3, S1=4, S2=5

  Pendidikan  Gaji_juta  Pendidikan_enc
2         SD          3             0.0
5        SMP          4             1.0
0        SMA          5             2.0
3         D3          8             3.0
1         S1         12             4.0
4         S2         18             5.0


# **4. Scaling & Normalisasi Fitur**
Feature scaling adalah proses menyamakan skala atau rentang nilai antar fitur numerik. Hal ini penting karena banyak algoritma machine learning sangat sensitif terhadap perbedaan besaran data; jika tidak diseragamkan, fitur dengan nilai besar akan mendominasi perhitungan dan menurunkan akurasi model.

**4.1 Mengapa Scaling Diperlukan?**

Tanpa scaling, fitur dengan rentang nilai luas (misalnya Pendapatan 0–50 juta) akan menenggelamkan pengaruh fitur bernilai kecil (misalnya Usia 0–100). Akibatnya, algoritma berbasis jarak seperti KNN menjadi bias karena menganggap selisih pendapatan sebagai satu-satunya faktor penentu, sementara perbedaan usia diabaikan.

**4.2 MinMaxScaler: Normalisasi ke [0, 1]**

MinMaxScaler mentransformasi setiap nilai ke dalam rentang [0, 1]

In [72]:
df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12] # dalam juta rupiah
})
scaler = MinMaxScaler(feature_range=(0, 1)) # default
# fit_transform: belajar min/max + transformasi data
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])
print('Min per fitur :', scaler.data_min_) # [25 5]
print('Max per fitur :', scaler.data_max_) # [55 35]
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_sc', 'Pendapatan_sc']).round(3))
# PENTING: gunakan .transform() saja pada data baru (test set!)
# X_test_scaled = scaler.transform(X_test)
# Balik ke skala asli
X_original = scaler.inverse_transform(X_scaled)

Min per fitur : [25.  5.]
Max per fitur : [55. 35.]

   Usia_sc  Pendapatan_sc
0    0.000          0.000
1    0.667          0.500
2    0.233          0.100
3    1.000          1.000
4    0.100          0.233


**4.3 StandardScaler: Z-score Standardization**

StandardScaler mengubah distribusi setiap fitur agar memiliki mean = 0 dan standar deviasi
= 1

In [73]:
df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12]
})
scaler = StandardScaler()
X = df[['Usia', 'Pendapatan']]
X_scaled = scaler.fit_transform(X)
print('Mean per fitur:', scaler.mean_) # rata-rata setiap kolom
print('Scale per fitur:', scaler.scale_) # std dev setiap kolom
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_z', 'Pend_z']).round(3))
# Contoh output:
# Usia_z Pend_z
# 0 -1.110 -0.784 (di bawah rata-rata)
# 3 1.533 1.568 (di atas rata-rata)
# Selalu .transform() saja pada test set
# X_test_z = scaler.transform(X_test)

Mean per fitur: [37. 16.]
Scale per fitur: [11.296017   10.75174404]

   Usia_z  Pend_z
0  -1.062  -1.023
1   0.708   0.372
2  -0.443  -0.744
3   1.593   1.767
4  -0.797  -0.372


**4.4 RobustScaler**

RobustScaler menggunakan median dan Interquartile Range (IQR) sebagai pengganti mean
dan standar deviasi, sehingga tidak terpengaruh oleh outlier ekstrem

In [74]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])
# Gunakan saat dataset mengandung outlier yang tidak bisa dihapus

---
# **6. Aktivitas Hands-on: Preprocessing Dataset Titanic**
---



**6.2 Langkah-Langkah Praktikum**

Langkah 1: Load & EDA Singkat
Muat dataset, periksa missing values, tipe data, dan distribusi target.

In [75]:
import pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
df = sns.load_dataset('titanic')
# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))
# survived=0: ~61.6%, survived=1: ~38.4% — kelas tidak seimbang!

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


kita muat dataset Titanic dan langsung melakukan seleksi kolom untuk membuang field yang tidak  digunakan. Hasil EDA singkat menunjukkan dua hal krusial: pertama, adanya missing values pada kolom age, embarked yang wajib ditangani; kedua, distribusi target survived tidak seimbang (imbang 61%:38%), yang berarti model nanti perlu diperhatikan agar tidak bias ke kelas mayoritas


---



**Langkah 2 — Handling Missing Values**

Isi nilai yang hilang sebelum encoding. Gunakan median untuk kolom numerik (robust terhadap outlier) dan modus untuk kolom kategorikal.

In [76]:
# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())
# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
print('Missing setelah handling:')
print(df.isnull().sum()) # Semua harus 0

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


di langkah ini, kita menangani missing values agar data siap diproses lebih lanjut. Untuk kolom numerik seperti age, kita gunakan median karena lebih tahan terhadap outlier dibanding mean. Sementara untuk kolom kategorikal embarked, diisi dengan modus (nilai yang paling sering muncul). Hasilnya, semua kolom kini bernilai 0 missing, artinya dataset sudah lengkap dan bersih


---



**Langkah 3: Encoding Kategorikal**

Terapkan One-Hot Encoding pada kolom 'sex' dan 'embarked'. Gunakan drop_first=True
untuk menghindari dummy variable trap.

In [77]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
columns=['sex', 'embarked'],
drop_first=True, # hindari dummy variable trap
dtype=int)
print('Kolom setelah encoding:')
print(df.columns.tolist())
# ['pclass','age','sibsp','parch','fare','survived',
# 'sex_male','embarked_Q','embarked_S']

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


Di langkah ini, kita menerapkan One-Hot Encoding pada kolom kategorikal sex dan embarked agar data bisa diproses oleh model machine learning.


---



**Langkah 4: Train-Test Split**

Bagi data dengan stratifikasi untuk menjaga proporsi kelas 'survived' di train dan test set.

In [78]:
from sklearn.model_selection import train_test_split
X = df.drop('survived', axis=1)
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(
X, y,
test_size=0.2,
random_state=42,
stratify=y # proporsi kelas terjaga
)
print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


kita bagi dataset menjadi dua bagian: training set untuk melatih model dan test set untuk evaluasi. Kita menggunakan parameter stratify=y agar proporsi kelas survived (0 dan 1) tetap seimbang di kedua set, mencegah bias saat pengujian.


---



**Langkah 5: Feature Scaling**

Terapkan StandardScaler HANYA pada kolom numerik, fit pada training set, transform pada
keduanya. Kolom biner hasil OHE tidak perlu di-scale.

In [79]:
from sklearn.preprocessing import StandardScaler
# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) TIDAK perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']
scaler = StandardScaler()
# fit_transform pada training set (belajar μ dan σ dari sini)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
# transform saja pada test set (gunakan μ dan σ dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])
print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))
print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8), y_test : (179,)


Di langkah ini, kita melakukan feature scaling menggunakan Standardscaler agar semua fitur numerik berada dalam skala yang seimbang. kita hanya menerapkan scaling pada kolom numerik (pclass, age, sibsp, parch, fare), sementara kolom biner hasil OHE dibiarkan apa adanya karena sudah bernilai 0 atau 1.


---



**Kesimpulan Kegiatan Hands-On**

Rangkaian kegiatan Praktikum membentuk fondasi dalam persiapan data untuk analisis dan pemodelan. Proses dimulai dengan eksplorasi awal dan pembersihan data (Langkah 1-3), di mana kita melakukan inspeksi dataset, menangani missing values, duplikat, serta inkonsistensi format untuk memastikan integritas data. Selanjutnya, dilakukan transformasi fitur melalui encoding kategorikal (Langkah 4) agar data dapat diproses oleh algoritma, serta scaling numerik (Langkah 5) untuk menstandarisasi skala antar variabel dan mencegah dominasi fitur tertentu.

kelima langkah ini menjamin bahwa data yang masuk ke tahap pemodelan telah bersih, dan optimal untuk dapat digunakan pada tahap selanjutnya